# Diagnóstico temporal — Optimización del predictor cíclico

La sensibilidad EMA `0.90` alineó mejor online y target, pero el predictor siguió sin recuperar el espectro. Repetimos exactamente esa corrida y registramos métricas por época, sin cambiar optimizer, checkpoint ni gates.

Medimos gradientes, movimiento desde `M₀`, error espectral, distancia al operador post-hoc online y desacople online/EMA. Estas métricas son diagnósticas: el checkpoint continúa seleccionado exclusivamente por mínima validation loss total y test no se construye.

In [ ]:
# ruff: noqa: E402, E501
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display

from koopman_jepa.config import DataConfig, ExperimentConfig, ModelConfig, TrainConfig
from koopman_jepa.koopman import expected_active_spectrum, match_eigenvalues
from koopman_jepa.model import TemporalJEPA
from koopman_jepa.phase_analysis import evaluate_phase_operator_diagnostics
from koopman_jepa.phase_data import PhaseWindowConfig, make_phase_tensor_dataset_splits
from koopman_jepa.training import collect_paired_embeddings, select_device, set_seed, train_model_with_validation_checkpoint

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
CONFIG_PATH = ROOT / "configs" / "stage3_cyclic_neural_ema_fast_smoke.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    raw = yaml.safe_load(handle)
dynamics = raw["dynamics"]
base_seed = int(raw["base_seed"])
emission_config = PhaseWindowConfig(**raw["emission"], repeats_per_transition=1)
model_raw = raw["model"]
experiment_config = ExperimentConfig(
    data=DataConfig(context_length=emission_config.window_length),
    model=ModelConfig(latent_dim=model_raw["latent_dim"], channels=model_raw["channels"], predictor_init=model_raw["predictor_init"]),
    train=TrainConfig(seed=base_seed, **raw["train"]),
)
splits = make_phase_tensor_dataset_splits(
    emission_config,
    train_repeats_per_transition=raw["splits"]["train_repeats_per_transition"],
    validation_repeats_per_transition=raw["splits"]["validation_repeats_per_transition"],
    seed=base_seed,
)
train_dataset = splits.train[dynamics]
validation_dataset = splits.validation[dynamics]
print(json.dumps({"replayed_config": CONFIG_PATH.name, "checkpoint_metric": raw["selection"]["metric"], "test_constructed": False}, indent=2))

In [ ]:
set_seed(base_seed)
device = select_device(experiment_config.train.device)
model = TemporalJEPA(
    latent_dim=model_raw["latent_dim"],
    channels=model_raw["channels"],
    predictor_init=model_raw["predictor_init"],
    pooling=model_raw["pooling"],
    input_length=emission_config.window_length,
).to(device)
initial_predictor = model.predictor.matrix.detach().cpu().numpy().copy()
initial_norm = np.linalg.norm(initial_predictor)
expected_spectrum = expected_active_spectrum(dynamics, emission_config.num_phases)

def trace_epoch(epoch, current_model, row):
    current, future_online, future_target, phase_pairs = collect_paired_embeddings(
        current_model, validation_dataset, experiment_config.train.batch_size, device
    )
    matrix = current_model.predictor.matrix.detach().cpu().numpy()
    diagnostics = evaluate_phase_operator_diagnostics(
        current, future_online, future_target, phase_pairs[:, 0], phase_pairs[:, 1], matrix, dynamics
    )
    spectral_match = match_eigenvalues(np.linalg.eigvals(matrix), expected_spectrum)
    return {
        "trace_predictor_relative_movement": float(np.linalg.norm(matrix - initial_predictor) / initial_norm),
        "trace_predictor_spectral_max_error": spectral_match["max_absolute_error"],
        "trace_predictor_online_error": diagnostics["trained_online_endomorphism_error"],
        "trace_predictor_posthoc_distance": diagnostics["predictor_vs_posthoc_online_error"],
        "trace_posthoc_online_spectral_max_error": diagnostics["posthoc_online_spectral_max_error"],
        "trace_online_target_basis_error": diagnostics["online_target_phase_basis_error"],
    }

training_result = train_model_with_validation_checkpoint(
    model, train_dataset, validation_dataset, experiment_config, device, epoch_callback=trace_epoch
)
history = training_result.history
assert training_result.best_epoch == 27

In [ ]:
selected_index = training_result.best_epoch - 1
best_spectral_index = int(np.argmin([row["trace_predictor_spectral_max_error"] for row in history]))
best_posthoc_index = int(np.argmin([row["trace_posthoc_online_spectral_max_error"] for row in history]))
predictor_gradients = np.array([row["train_predictor_gradient_norm"] for row in history])
online_gradients = np.array([row["train_online_gradient_norm"] for row in history])
spectral_threshold = raw["gates"]["maximum_spectral_error"]
ever_passed_spectrum = bool(history[best_spectral_index]["trace_predictor_spectral_max_error"] <= spectral_threshold)
trace_summary = {
    "selected_epoch": training_result.best_epoch,
    "selected_predictor_spectral_error": history[selected_index]["trace_predictor_spectral_max_error"],
    "best_predictor_spectral_epoch": best_spectral_index + 1,
    "best_predictor_spectral_error": history[best_spectral_index]["trace_predictor_spectral_max_error"],
    "predictor_ever_passed_spectrum": ever_passed_spectrum,
    "selected_predictor_posthoc_distance": history[selected_index]["trace_predictor_posthoc_distance"],
    "selected_predictor_relative_movement": history[selected_index]["trace_predictor_relative_movement"],
    "maximum_predictor_relative_movement": max(row["trace_predictor_relative_movement"] for row in history),
    "mean_predictor_gradient_norm": float(predictor_gradients.mean()),
    "mean_online_gradient_norm": float(online_gradients.mean()),
    "mean_predictor_online_gradient_ratio": float(np.mean(predictor_gradients / np.maximum(online_gradients, 1e-15))),
    "best_posthoc_spectral_epoch": best_posthoc_index + 1,
    "best_posthoc_spectral_error": history[best_posthoc_index]["trace_posthoc_online_spectral_max_error"],
    "selected_online_target_basis_error": history[selected_index]["trace_online_target_basis_error"],
}
print(json.dumps(trace_summary, indent=2))

In [ ]:
epochs = np.arange(1, len(history) + 1)
fig, axes = plt.subplots(2, 2, figsize=(15, 11), constrained_layout=True)
axes[0, 0].plot(epochs, [row["val_loss"] for row in history], label="validation total")
axes[0, 0].plot(epochs, [row["val_prediction_loss"] for row in history], label="validation prediction")
axes[0, 0].axvline(training_result.best_epoch, color="black", linestyle=":", label="checkpoint")
axes[0, 0].set(title="Selección por validation", xlabel="Época", ylabel="Loss")
axes[0, 0].legend()

axes[0, 1].plot(epochs, [row["trace_predictor_spectral_max_error"] for row in history], label="predictor")
axes[0, 1].plot(epochs, [row["trace_posthoc_online_spectral_max_error"] for row in history], label="post-hoc online")
axes[0, 1].axhline(spectral_threshold, color="tab:red", linestyle="--", label="gate")
axes[0, 1].axvline(training_result.best_epoch, color="black", linestyle=":")
axes[0, 1].set(title="Error espectral por época", xlabel="Época", ylabel="Error máximo")
axes[0, 1].legend()

axes[1, 0].semilogy(epochs, online_gradients, label="encoder online")
axes[1, 0].semilogy(epochs, predictor_gradients, label="predictor")
axes[1, 0].set(title="Normas de gradiente pre-clipping", xlabel="Época", ylabel="Norma L2 media")
axes[1, 0].legend()

axes[1, 1].plot(epochs, [row["trace_predictor_relative_movement"] for row in history], label="movimiento desde M₀")
axes[1, 1].plot(epochs, [row["trace_predictor_posthoc_distance"] for row in history], label="distancia al post-hoc")
axes[1, 1].plot(epochs, [row["trace_online_target_basis_error"] for row in history], label="desacople online/EMA")
axes[1, 1].axvline(training_result.best_epoch, color="black", linestyle=":")
axes[1, 1].set(title="Evolución geométrica", xlabel="Época", ylabel="Error relativo")
axes[1, 1].legend()
plt.show()

selection_reading = (
    "El predictor cruzó el gate en otra época: la selección por loss merece revisión."
    if ever_passed_spectrum
    else "El predictor nunca cruzó el gate espectral: no es sólo un problema de selección del checkpoint."
)
display(Markdown(f"""## Lectura de la traza

- Checkpoint por validation: época **{training_result.best_epoch}**; su error espectral: **{history[selected_index]['trace_predictor_spectral_max_error']:.3f}**.
- Mejor época espectral del predictor: **{best_spectral_index + 1}**, error **{history[best_spectral_index]['trace_predictor_spectral_max_error']:.3f}**.
- El predictor cruzó alguna vez el gate espectral: **{'sí' if ever_passed_spectrum else 'no'}**.
- Gradiente medio predictor / encoder: **{predictor_gradients.mean():.3f} / {online_gradients.mean():.3f}**.
- Movimiento de `M` en el checkpoint / máximo: **{history[selected_index]['trace_predictor_relative_movement']:.3f} / {max(row['trace_predictor_relative_movement'] for row in history):.3f}**.
- Distancia predictor/post-hoc en el checkpoint: **{history[selected_index]['trace_predictor_posthoc_distance']:.3f}**.
- Mejor error espectral post-hoc online: **{history[best_posthoc_index]['trace_posthoc_online_spectral_max_error']:.3f}**.

{selection_reading}

Esta traza no cambia el `FAIL` ni selecciona modelos con información espectral. Sirve para decidir el próximo control de optimización sin consultar test.
"""))